# Sub-1B document-VLM comparison on a T4 GPU

Thin runner: **clone → install → run repo scripts**. Runs the established sub-1B chat VLMs, **PaddleOCR-VL 1.0/1.5/1.6**, and the **newer 2025-26 releases** — **MiniCPM-V-4.6, LFM2.5-VL-1.6B, Qwen3.5-VL-0.8B, LightOnOCR-1B** — on the capability, spatial/context, and **proposed custom-eval** sets, measuring **score + inference time + CPU/GPU memory**.

Each `(model × benchmark)` run prints a global progress line `[done/total] (N left) stage: model × bench`, so it is always clear how many experiments remain and which stage is running. Tables below render as clean Markdown (not raw pipe-text) plus a sorted bar chart.

Runtime → Change runtime type → **T4 GPU**, then Run all.

## 1. GPU check

In [1]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-3e613b6d-1cff-9174-00c4-62f6453873b2)


## 2. Clone repo + install

In [2]:
%cd /content
![ -d OCR ] || git clone https://github.com/SangbumChoi/OCR.git
%cd /content/OCR
!git checkout claude/new-session-w79q0i && git pull --ff-only
!pip -q install -e '.[models,finetune]' protobuf

/content
Cloning into 'OCR'...
remote: Enumerating objects: 1682, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 1682 (delta 64), reused 131 (delta 46), pack-reused 1509 (from 1)
Receiving objects: 100% (1682/1682), 55.48 MiB | 30.71 MiB/s, done.
Resolving deltas: 100% (645/645), done.
/content/OCR
Branch 'claude/new-session-w79q0i' set up to track remote branch 'claude/new-session-w79q0i' from 'origin'.
Switched to a new branch 'claude/new-session-w79q0i'
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:0

## 3. Run the full comparison (3 passes; all measured, with progress logging)
Installs CJK fonts + QR/barcode libs, builds the probes incl. the custom-eval set, then runs three passes that pin different `transformers` versions: **pass 1** chat VLMs @ tf4.49, **pass 2** PaddleOCR-VL @ tf4.57, **pass 3** the newer 2025-26 VLMs @ latest. Per-model failures are captured as run-status data, and a `[done/total]` counter reports remaining experiments.

In [3]:
!DEVICE=cuda bash scripts/run_full_comparison.sh

== deps for the custom set (CJK fonts + QR/barcode) ==
== probes ==
[done] 6 capability samples -> data/probes/capability_probe/capability.jsonl
   T1                   metric=anls         gold=INV-2025-0042
   T2                   metric=anls         gold=Acme Corporation
   H1                   metric=relaxed_acc  gold=145.50
   H2                   metric=exact        gold=Gadget B
   H3                   metric=relaxed_acc  gold=70
   L1                   metric=grounding    gold=40,334,222,355;820,600
[done] 15 samples -> data/probes/spatial_context_probe/probe.jsonl
   sp_quad_top-left           spatial-quadrant       gold=top-left ctrl=False
   sp_quad_top-right          spatial-quadrant       gold=top-right ctrl=False
   sp_quad_bottom-left        spatial-quadrant       gold=bottom-left ctrl=False
   sp_quad_bottom-right       spatial-quadrant       gold=bottom-right ctrl=False
   sp_relpos_normal           spatial-relative       gold=below ctrl=False
   sp_relpos_counterfactua

In [ ]:
# --- pretty rendering helpers (clean tables + a readable bar chart) ---
from pathlib import Path
import json
from IPython.display import Markdown, display

def show_md(path):
    """Render a generated markdown table as a real (aligned) table instead of raw pipe-text."""
    p = Path(path)
    display(Markdown(p.read_text() if p.exists() else f"_missing: {path}_"))

def bar_capability(json_path="docs/results/matrix_capability.json"):
    """Horizontal bar chart of each model's MEAN capability score (sorted, labels left-aligned)."""
    import matplotlib.pyplot as plt
    p = Path(json_path)
    if not p.exists():
        print("no matrix yet:", json_path); return
    j = json.loads(p.read_text())
    scores = j.get("scores", {})
    means = {m: (sum(v.values())/len(v)) for m, v in scores.items() if v and m != "dummy-echo"}
    if not means:
        print("no scores to plot yet"); return
    items = sorted(means.items(), key=lambda kv: kv[1])
    labels = [k for k, _ in items]; vals = [v for _, v in items]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45*len(labels))))
    bars = ax.barh(labels, vals, color="#3a7bd5")
    ax.set_xlim(0, 1.0); ax.set_xlabel("mean capability score (T1/T2/H1/H2/H3/L1)")
    ax.set_title("Mean capability-probe score by model")
    ax.tick_params(axis="y", labelsize=10)
    for b, v in zip(bars, vals):
        ax.text(min(v + 0.02, 0.96), b.get_y() + b.get_height()/2, f"{v:.2f}",
                va="center", fontsize=9)
    plt.tight_layout(); plt.show()


## 4. Scores + efficiency (time & memory)

In [ ]:
show_md('docs/results/matrix_capability.md')
bar_capability()

## 5. Spatial / context shortcut-robust signals

In [ ]:
show_md('docs/results/matrix_probe.md')
show_md('docs/results/probe_signals.md')

## 6. Proposed custom-eval — by class / language / rotation / direction / spotting

In [ ]:
show_md('docs/results/custom_eval_breakdown.md')

## 7. PaddleOCR-VL 1.0 vs 1.5 vs 1.6

In [ ]:
# PaddleOCR-VL 1.0 vs 1.5 vs 1.6 — score / latency / memory as a clean table
import json
from pathlib import Path
from IPython.display import Markdown, display
rows = ["| version | score | avg latency (s) | peak GPU (MB) |", "| --- | --- | --- | --- |"]
for m in ["paddleocr-vl", "paddleocr-vl-1.5", "paddleocr-vl-1.6"]:
    f = Path(f"docs/results/{m}/capability/summary.json")
    if f.exists():
        d = json.loads(f.read_text())
        rows.append(f"| {m} | {d.get('score')} | {d.get('avg_latency_s')} | {d.get('peak_gpu_mb')} |")
    else:
        rows.append(f"| {m} | _n/a_ | _n/a_ | _n/a_ |")
display(Markdown("\n".join(rows)))

## 8. Download all results

In [8]:
!zip -qr /content/docvlm_results.zip results
from google.colab import files; files.download('/content/docvlm_results.zip')


zip error: Nothing to do! (try: zip -qr /content/docvlm_results.zip . -i results)


FileNotFoundError: Cannot find file: /content/docvlm_results.zip